# RNN 基础

这块可以参考之前的 RNN 笔记 （神经网络基础部分），在 NLP 这块目前重点介绍 seq2seq 架构的 RNN 模型




<img src="./markdown-note/images/img_7.png" width="600"/>

```text
我们把目光集中在中间的方块部分，它的输入有两部分，分别是h（t-1）以及x（t)，代表上一时间步的隐藏层输出，以及此时间步的输入

它们进入RNN结构体后，会融合“到一起，这种融合我们根据结构解释可知，是将二者进行拼接，形成新的张量 [x（t），h（t-1）]

之后这个新的张量将通过一个全连接层（线性层），该层使用tanh作为激活函数，最终得到该时间步的输出h（t）

它将作为下一个时间步的输入和x（t+1）一起进入结构体.以此类推
```

拆开 RNN 处理单元之后的示意图如下

<img src="./markdown-note/images/img_8.png" width="800">


In [ ]:
from unicodedata import category

import torch.nn as nn
import torch
from torch.nn.functional import batch_norm


def rnn_demo1():
   # 第一个参数：input_size = 输入张量 x 的维度，简单理解就是每个词向量的维度
   # 第二个参数：hidden_size = 隐藏层输出 h 的维度 这个参数是可以自己定义
   # 第三个参数：num_layers=RNN 结构体包含的隐藏层层数，一般情况都是 1
    rnn = nn.RNN(5, 6, 1)

    # sequence_length: 输入序列长度,也就是句子的长度
    # batch_size: 批次样本的数量
    # input_size: 输入张量 x 的维度，也就是每个词的维度
    input=torch.randn(6, 3, 5)

   # 初始化隐藏层
   # 参数 1：num_layers =RNN 结构体包含的层数
   # 参数 2：batch_size=批次样本的数量
   # 参数 3：hidden_size=隐藏层输出 h 的维度 也就是隐藏层的神经元个数
    h0=torch.randn(1, 3, 6)

    # 这里相当于调用了 forward 方法，h0可以不写
    # output: 所有时间步的最后一层隐藏状态的集合 形状：(seq_len,batch_size,hidden_size）
    # hn: 最后一个时间步（t=20）的所有层隐藏状态的集合 形状：(num_layers,batch_size,hidden_size）
    output,hn=rnn(input,h0)

    print("output=",output.shape)
    print("hn=",hn.shape)
    print("output=",output)
    print("hn=",hn)
    #print("rnn=",output)

rnn_demo1()

**第一个返回值：output**
本质：所有时间步的最后一层隐藏状态的集合。

形状:(seq_len,batch_size,hidden_size）

通俗解释：把每个时间步 t=1，t=2,...，t=20 的**最后一层RNN单元输出**的h_t按时间顺序排成一个序列，就是output

**第二个返回值：hn**

本质：最后一个时间步（t=20）的所有层隐藏状态的集合。

形状: (num_layers, batch_size, hidden_size)→ (2, 32, 128)

通俗解释：只取最后一个时间步（t=20），把这一时刻“第一层的h_20^1"、“第二层的h_20~2“按层数顺序排列，就是hn.
比如 hn[0] 是t=20 时刻第一层的 h_20^1，hn[1]是t=20时刻第二层的h_20^2。

所以你会发现在上面的代码中最终输出的 output 的最后一个张量 和 hn 是一样的


在创建 RNN 的时候 ` rnn = nn.RNN(5, 6, 1,batch_first=True)` 可以指定 batch_first 参数，这个参数代表的含义是：

- 如果设置为 True，则输入和输出的张量使用的是 (batch_size, sequence_length, input_size) 的格式。
- 如果设置为 False（默认），则使用 (sequence_length, batch_size, input_size) 的格式。

一句话总结就是要不要在输入的时候把批次大小放到第一个参数位置上





**tips**: 请问 nn.RNN 执行之后应该返回的是 RNN 的实例 ，为什么这个实例能够直接运行，此外神经网络又是什么时候执行的 forward 方法呢？


实际的执行逻辑是这样的：
- 创建 RNN 实例 → 指向 RNNBase
- 调用 rnn(input, h0) → 实际调用 `__call__() `
- `__call__()` → 调用 forward(input, h0)
- forward() → 执行 RNN 的实际计算逻辑
- nn.RNN 并不是直接定义了 forward()，而是继承自 torch.nn.modules.rnn.RNNBase，这个基类中才真正实现了 forward() 方法
- `__call__()` 是 torch.nn.Module 的方法。

# LSTM
传统 RNN 在训练过程中使用反向传播（BPTT，Backpropagation Through Time）时，会发现：

当序列很长时，梯度在多次乘积后容易消失（vanish） 或 爆炸（explode）

这导致模型很难学习到长距离依赖关系

举个例子：
如果你要翻译一个英文句子，比如 "The cat, which was sitting on the mat, is sleeping."

RNN 可能无法记住前面的 "cat" 和 "mat"，从而影响后面生成 "sleeping" 的准确性。




LSTM（Long Short-TermMemory）也称长短时记忆结构，它是传统RNN的变体，与经典RNN相比能够有效捕捉长序列之间的语义关联，**缓解梯度消失或爆炸现象**.
同时LSTM的结构更复杂，它的核心结构可以分为四个部分去解析 遗忘门 、输入门 、细胞状态、 输出门

<img src="./markdown-note/images/img_9.png" width="600"/>

## 遗忘门
<img src="./markdown-note/images/img_10.png" width="400"/>
与传统RNN的内部结构计算非常相似，首先将当前时间步输入x(t)与上一个时间步隐含状态h(t-1)拼接 得到[x(t),h(t-1)]通过一个全连接层做变换，最后通过sigmoid函数进行激活得到f(t)

我们可以将f(t)看作是门值，好比一扇门开合的大小程度，门值都将作用在通过该扇门的张量  遗忘门决定了从上一时刻的细胞状态中丢弃哪些信息 

遗忘门的门值是由x（t）和 h（t-1）计算得来的，因此整个公式意味着**根据当前时间步输入和上一个时间步隐含状态h（t-1）来决定遗忘多少上一层的细胞状态所携带的过往信息**

## 输入门
<img src="./markdown-note/images/img_11.png" width="400"/>

我们看到输入门的计算公式有两个，两个公式使用的激活函数不同
第一个就是产生输入门门值的公式，它和遗忘门公式几乎相同，区别只是在于它们之后要作用的目标上.这个公式意味着输入信息有多少需要进行过滤. 也就是说输入门决定将当前输入中的哪些新信息存入细胞状态。
输入门的第二个公式是与传统RNN的内部结构计算相同
对于LSTM来讲，它得到的是当前的细胞状态，而不是像经典RNN一样得到的是隐含状态

## 细胞状态
<img src="./markdown-note/images/img_12.png" width="400"/>
细胞更新的结构与计算公式非常容易理解，这里没有全连接层，只是将刚刚得到的遗忘门门值 ft 与上一个时间步得到的C（t-1）相乘，再加上输入门门值与当前时间步得到的未更新 C（t）相乘的结果.最终得到更新后的C（t)作为下一个时间步输入的一部分

整个细胞状态更新过程就是对遗忘门和输入门的应用

## 输出门
<img src="./markdown-note/images/img_13.png" width="400"/>
输出门部分的公式也是两个，第一个即是计算输出门的门值，它和遗忘门，输入门计算方式相同.

第二个即是使用这个门值产生隐含状态h（t)，他将作用在更新后的细胞状态C（t）上，并使用tanh激活，

最终得到h（t)作为下一一时间步输入的一部分.整个输出门的过程，就是为了产生隐含状态h（t） 也就是说 输出门决定了根据当前的细胞状态，输出哪些信息到隐藏状态。

## BI-LSTM

<img src="./markdown-note/images/img_14.png" width="600"/>

我们看到图中对”我爱中国"这句话或者叫这个输入序列进行了从左到右和从右到左两次LSTM处理，将得到的结果张量进行了拼接作为最终输出
这种结构能够捕捉语言语法中一些特定的前置或后置特征，增强语义关联，但是模型参数和计算复杂度也随之增加了一倍，一般需要对语料和计算资源进行评估后决定是否使用该结构

In [ ]:
import torch.nn as nn
import torch

input_size=5 # 输入词向量维度
hidden_size=6 # 隐藏层输出 h 的维度 这个参数是可以自己定义
num_layers=1
batch_size=2
sequence_length=3
# 设置 bidirectional=True 即可实现双向LSTM
gru = nn.LSTM(input_size, hidden_size, num_layers)
input = torch.randn(sequence_length, batch_size, input_size)
h0 = torch.randn(num_layers, batch_size, hidden_size)
c0 = torch.randn(num_layers, batch_size, hidden_size)

# h0 和 c0 可以不传，查看源码可知，如果不传就会设置为全 0 张量
output, (hn, cn) = gru(input, (h0, c0))
print(output.shape, hn.shape, cn.shape)

# 形状:(seq_len, batch_size, hidden_size）
print("output->",output)
# 形状:(num_layers, batch_size, hidden_size）
print("hn->",hn)
# 形状:(num_layers, batch_size, hidden_size）
print("cn->",cn)

# GRU介绍
GRU（Gated Recurrent Unit）也称门控循环单元结构，它也是传统RNN的变体，同LSTM一样能够有效捕捉长序列之间为语义关联，缓解梯度消失或爆炸现象，同时它的结构和计算要比LSTM更简单，它的核心结构可以分为两个部分去解析：
- 更新门：决定新旧信息之间如何混合。
- 重置门：决定要忘记之前的信息有多少。

<img src="./markdown-note/images/img_16.png" width="600"/>

<img src="./markdown-note/images/img_17.png" width="500"/>

和之前分析过的LSTM中的门控一样，首先计算更新门和重置门的门值，分别是z（t）和r（t)，计算方法就是使用x（t）与h（t-1）拼接进行线性变换，再经过sigmoid激活.

之后重置门门值作用在了h（t-1）上，代表控制上一时间步传来的信息有多少可以被利用

接着就是使用这个重置后的h（t-1）进行基本的RNN计算，即与x（t)拼接进行线性变化，经过tanh激活，得到新的h（t).

最后更新门的门值会作用在新的h（t），而1-门值会作用在h（t-1）上，随后将两者的结果相加得到最终的隐含状态输出h（t)，这个过程意味着更新门有能力保留之前的结果，当门值趋于1
时，输出就是新的h（t），而当门值趋于0时，输出就是上一时间步的h（t-1）

In [ ]:
import torch.nn as nn
import torch

input_size=5 # 输入词向量维度
hidden_size=6 # 隐藏层输出 h 的维度 这个参数是可以自己定义
num_layers=1
batch_size=2
sequence_length=3

gru = nn.GRU(input_size, hidden_size, num_layers)

input = torch.randn(sequence_length, batch_size, input_size)
h0 = torch.randn(num_layers, batch_size, hidden_size)

output, hn = gru(input, h0)

# 形状:(seq_len, batch_size, hidden_size）
print("output->",output)
# 形状: (num_layers, batch_size, hidden_size)
print("hn->",hn)

## 人名分类案例

### 导入依赖包

In [ ]:
#导入torch工具
import torch
#导入nn准备构建模型
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
#导入torch的数据源数据迭代器工具包
from torch.utils.data import Dataset, DataLoader
#用于获得常见字母及字符规范化
import string
#导入时间工具包
import time
#引入制图工具包
import matplotlib.pyplot as plt
#从io中导入文件打开方法
from io import open

### 准备数据集 DataSet

In [ ]:
def read_name_file():
    name_list = []
    country_list = []

    with open("./data/names.txt", 'r', encoding='utf-8') as f:
        for line in f:

            # split() 会自动处理多个连续空格
            parts = line.strip().split()

            # 确保每行至少有两个部分（姓名和国家）
            if len(parts) >= 2:
                name = parts[0]
                country = parts[1]
                name_list.append(name)
                country_list.append(country)

    return name_list, country_list

In [ ]:
class NameClassDataSet(Dataset):
    # 类变量 letters 存储所有字母 总共 57个
    letters = string.ascii_letters + " .,;'"
    category = ['Italian', 'English', 'Arabic', 'Spanish', 'Scottish', 'Irish', 'Chinese', 'Vietnamese', 'Japanese',
                'French', 'Greek', 'Dutch', 'Korean', 'Polish', 'Portuguese', 'Russian', 'Czech', 'German']

    def __init__(self):
        name_list, country_list = read_name_file()
        self.name_list = name_list
        self.country_list = country_list
        self.len = len(name_list)

    def __len__(self):
        return self.len

    def __getitem__(self, idx):
        # 对索引进行边界检查
        idx = min(max(idx, 0), self.len - 1)
        x = self.name_list[idx]
        y = self.country_list[idx]
        # 对于每个名字采用 oneHot 编码标识，一个名字有多个字母，每个字母采用 onehot 编码
        # 矩阵形状是 人名长度 * 字母个数 的矩阵  每一行只有一个数值为 1
        tensor_x = torch.zeros(len(x), len(NameClassDataSet.letters))
        for i, letter in enumerate(x):
            # 找到 x 也就是人名里面每个字母的索引 标记为 1 实现 onehot 编码
            tensor_x[i][NameClassDataSet.letters.find(letter)] = 1
        tensor_y = torch.tensor(self.category.index(y), dtype=torch.long)
        return tensor_x, tensor_y


In [ ]:
read_name_file()

nameClassDataset =  NameClassDataSet(*read_name_file())
print(nameClassDataset[0][0].shape) # 8 * 57
print(nameClassDataset[0][1].shape) # 标量
# # DataLoader 是对 DataSet 的封装，实际上是一个迭代器
# # 所以可以使用 for 循环进行遍历，遍历出来的每个元素，就是 DataSet  __getitem__  方法返回的信息
# # 但是 DataLoader 因为支持分批 所以对 DataSet  __getitem__  方法返回的信息 进行了升维 也就是加了一个 batch_size 维度
# # 如果使用 len(dataLoader) 获取长度 = len(dataSet) / batchSize
dataLoader = DataLoader(nameClassDataset, batch_size=1, shuffle=True)

for x,y in dataLoader:
    print(x.shape)  # batch_size * 8 * 57
    print(y.shape)  # batch_size
    break

### 构建模型

In [ ]:
import torch
#导入nn准备构建模型
import torch.nn as nn
class NameClassifyGRU(nn.Module):
    def __init__(self,input_size,hidden_size,output_size,num_layers=1):
        super(NameClassifyGRU, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers
        # GRU 层
        self.gru = nn.GRU(input_size, hidden_size, num_layers)
        self.linear = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=-1)

    def forward(self, input, hidden):
        # gru 要求的 input 的 shape = [sequence_length , batch_size, input_size]
        # sequence_length: 输入序列长度,也就是句子的长度
        # batch_size: 批次样本的数量
        # input_size: 输入张量 x 的维度，也就是每个词的维度
        # 原始的 input 形状是 [sequence_length , 57]，需要补齐 batch_size
        # [sequence_length , 57] -> [sequence_length ,1, 57]
        input = input.unsqueeze(1)
        output,hn = self.gru(input, hidden)
        # 获取最后一个时间步的输出
        temp = output[-1]
        return self.softmax(self.linear(temp)), hn

    def init_hidden(self):
       # 初始化隐藏层
       # 参数 1：num_layers =RNN 结构体包含的层数
       # 参数 2：batch_size=批次样本的数量
       # 参数 3：hidden_size=隐藏层输出 h 的维度 也就是隐藏层的神经元个数
        return torch.zeros(self.num_layers, 1, self.hidden_size)

### 模型训练

In [ ]:
def train_gru():
    name_class_dataset =  NameClassDataSet()
    data_loader = DataLoader(name_class_dataset, batch_size=1, shuffle=True)
    input_size = len(NameClassDataSet.letters) # 57
    n_hidden = 128
    output_size = len(NameClassDataSet.category) # 多少个国家 目前数据集中 18个国家 对应的分类结果有 18种

    model = NameClassifyGRU(input_size,n_hidden,output_size)
    # 损失函数
    loss = nn.NLLLoss()
    # 优化器 adam 梯度下降
    adam = optim.Adam(model.parameters(), lr=0.001)
    epochs=1

    for epoch in range(epochs):
        total_loss = 0
        predict_right_count = 0
        for i,(x,y) in enumerate(data_loader):
            out,hn = model(x[0], model.init_hidden())
            cur_loss = loss(out,y)

            adam.zero_grad()
            cur_loss.backward()
            adam.step()

            total_loss += cur_loss.item()
            if out.argmax(1).item() == y:
                predict_right_count += 1

            if i % 5 == 0 and i != 0:
                avg_loss = total_loss / i
                avg_accuracy = predict_right_count / i
                print("epoch:%d, avg_loss:%.3f, avg_accuracy:%.3f" % (epoch, avg_loss, avg_accuracy))

    torch.save(model.state_dict(), "name_classify_gru.pth") # 保存模型

train_gru()

**损失函数为什么是 NLLLoss？？**

在之前介绍神经网络基础的时候我们介绍过多分类问题使用交叉熵损失函数 当时使用的是 nn.CrossEntropyLoss() 这个损失函数会自动对原始输出计算 softmax 不需要人为的去添加 softmax 层，简单来说如果这里需要使用 nn.CrossEntropyLoss() 来计算损失我们就不能在设计模型的时候 给 Linear 层的输出再套 softmax 损失函数处理。


这里使用的则是另一种方案：使用 nn.NLLLoss()计算损失  上面的代码在网络的输出层已经计算了 `log(softmax(f(x)))`  所以可以理解为

**LogSoftmax + NLLLoss  等价于 无激活函数 + CrossEntropyLoss**


再有，其实我们知道交叉熵损失是 ` - ∑(yi * log(softmax(f(xi))))`  其中 yi 是 真实标签的 oneHot 编码，但是我们在pytorch 里面不需要去实际的做编码，**只需把真实目标值类别的索引对应的 log(softmax(f(x))) 值再取个负数作为损失函数的输出值即可**


In [ ]:
# 交叉熵损失demo
# 对于真实值可以使用oneHot编码 也可以不编码
# onehot编码的真实值
# y_true=torch.tensor([[0,1,0],[0,0,1]],dtype=torch.float32)
# 也可以使用未编码的真实值，必须是int64类型，表示的是样本属于哪个分类
y_true=torch.tensor([1,2],dtype=torch.int64)
y_pred=torch.tensor([[12,20,3],[9,2,14]],dtype=torch.float32)

# CrossEntropyLoss 会同时执行softmax和损失计算
loss = nn.CrossEntropyLoss()
l1=loss(y_pred,y_true).numpy()
print("CrossEntropyLoss",l1)

logsoftmax = nn.LogSoftmax(dim=-1)
loss = nn.NLLLoss()
l2=loss(logsoftmax(y_pred),y_true).numpy()
print("NLLLoss",l2)


# 可以发现 NLLLoss + LogSoftmax 损失函数 等价于 CrossEntropyLoss

### 模型预测

In [ ]:
def predict_by_gru(x):
    input_size = len(NameClassDataSet.letters) # 57
    n_hidden = 128
    output_size = len(NameClassDataSet.category) # 多少个国家 目前数据集中 18个国家 对应的分类结果有 18种

    # 声明模型
    gru = NameClassifyGRU(input_size, n_hidden, output_size)
    # 加载模型参数
    gru.load_state_dict(torch.load("name_classify_gru.pth"))

    # x 是一段文本需要改为张量 例如 deng 应该转为 4*57 的张量形式
    tensor_x = torch.zeros(len(x),len(NameClassDataSet.letters))
    for i,c in enumerate(x):
        tensor_x[i][NameClassDataSet.letters.find(c)] = 1

    # 预测的时候不需要做反向传播 和梯度更新
    with torch.no_grad():
        output,hn=gru(tensor_x, gru.init_hidden())
        # print(output.shape) # torch.Size([1, 18])
        # 从预测结果里面获取前 3 个概率最大的 排序依据是第 1 维， True 表示取最大值
        topn = 3
        topv,topi=output.topk(topn,1,True)
        print("x=", x)
        for i in range(topn):
            print(NameClassDataSet.category[topi[0][i]],topv[0][i])